Setup:

In [10]:
import sys
import os
import plotly.graph_objects as go
import requests
import pandas as pd

from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, os.path.dirname(os.getcwd()))

Practice pulling validator set from Polygon API and manually calculate the Nakamoto coefficient by hand

In [11]:
polygon_api = "https://staking-api.polygon.technology/api/v2/validators"
data = requests.get(polygon_api).json()

df = pd.DataFrame(data['result'])
df['stake'] = pd.to_numeric(df['delegatedStake'])

df = df.sort_values('stake', ascending=False).reset_index(drop=True)

total_stake = df['stake'].sum()
threshold = total_stake * 0.51

df['cumulative'] = df['stake'].cumsum()

nakamoto_coeff = (df['cumulative'] < threshold).sum() + 1

print("Total Stake:", total_stake)
print("51% Threshold:", threshold)
print("Nakamoto Coefficient:", nakamoto_coeff)

Total Stake: 3.589868301008615e+27
51% Threshold: 1.8308328335143938e+27
Nakamoto Coefficient: 7


Pull 2024 U.S. election price history from Polymarket and plot it

In [12]:
# Find condition ID for the presidental election
slug = "presidential-election-winner-2024"
response = requests.get(f"https://gamma-api.polymarket.com/events/slug/{slug}")
data = response.json()

# Trump market
condition_id = "0xdd22472e552920b8438158ea7238bfadfa4f736aa4cee91a6b86c39ead110917"
clob_market = requests.get(f"https://clob.polymarket.com/markets/{condition_id}").json()

# Yes token
token_id = "21742633143463906290569050155826241533067272736897614950488156847949938836455"

r = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={
        "market":   token_id,
        "interval": "max",
        "fidelity": 720,
    }
).json()

df_polymarket = pd.DataFrame(r['history'])
df_polymarket['t'] = pd.to_datetime(df_polymarket['t'], unit='s')
df_polymarket['p'] = df_polymarket['p'].astype(float)
df_polymarket = df_polymarket.sort_values('t').reset_index(drop=True)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_polymarket['t'],
    y=df_polymarket['p'],
    mode='lines',
    name='Trump Yes price',
    line=dict(color='royalblue', width=2)
))

fig.add_vline(
    x=pd.Timestamp('2024-11-05').timestamp() * 1000,
    line_dash='dash',
    line_color='red',
    annotation_text='Election Day',
    annotation_position='top right'
)

fig.update_layout(
    title='Trump Win Probability — Polymarket 2024 Presidential Election',
    xaxis_title='Date',
    yaxis_title='Implied Probability',
    yaxis=dict(range=[0, 1], tickformat='.0%'),
    hovermode='x unified'
)

fig.show()

Pull same election contract from Kalshi and overlay on Polymarket chart 

In [13]:
kalshi_url = "https://api.elections.kalshi.com/trade-api/v2"

r = requests.get(
    f"{kalshi_url}/historical/markets/PRES-2024-DJT/candlesticks",
    params={
        "start_ts": 1704067200,  # Jan 1 2024
        "end_ts":   1731542400,  # Nov 14 2024
        "period_interval": 1440, # 1 day in minutes
    }
).json()

candles = r['candlesticks']
df_kalshi = pd.DataFrame([{
    'timestamp': pd.Timestamp(c['end_period_ts'], unit='s', tz='UTC'),
    'price':     float(c['price']['close'])
} for c in candles if c['price']['close'] is not None])

df_kalshi = df_kalshi.sort_values('timestamp').reset_index(drop=True)

# Overlay both on same chart
fig = go.Figure()

# Polymarket line
fig.add_trace(go.Scatter(
    x=df_polymarket['t'],
    y=df_polymarket['p'],
    mode='lines',
    name='Polymarket',
    line=dict(color='royalblue', width=2)
))

# Kalshi line
fig.add_trace(go.Scatter(
    x=df_kalshi['timestamp'],
    y=df_kalshi['price'],
    mode='lines',
    name='Kalshi',
    line=dict(color='firebrick', width=2)
))

fig.add_vline(
    x=pd.Timestamp('2024-11-05').timestamp() * 1000,
    line_dash='dash',
    line_color='grey',
    annotation_text='Election Day',
    annotation_position='top right'
)

fig.update_layout(
    title='Trump Win Probability — Polymarket vs Kalshi',
    xaxis_title='Date',
    yaxis_title='Implied Probability',
    yaxis=dict(range=[0, 1], tickformat='.0%'),
    hovermode='x unified'
)

fig.show()

Focus on just the period of election week to get a closer view of price changes as the election is called

In [14]:
# zoom into the election window to see it clearly
fig_zoom = go.Figure()

# filter to just election week
mask_pm = (df_polymarket['t'] >= '2024-11-01') & (df_polymarket['t'] <= '2024-11-10')
mask_k  = (df_kalshi['timestamp'] >= '2024-11-01') & (df_kalshi['timestamp'] <= '2024-11-10')

fig_zoom.add_trace(go.Scatter(
    x=df_polymarket[mask_pm]['t'],
    y=df_polymarket[mask_pm]['p'],
    mode='lines+markers',
    name='Polymarket',
    line=dict(color='royalblue', width=2)
))

fig_zoom.add_trace(go.Scatter(
    x=df_kalshi[mask_k]['timestamp'],
    y=df_kalshi[mask_k]['price'],
    mode='lines+markers',
    name='Kalshi',
    line=dict(color='firebrick', width=2)
))

fig_zoom.add_vline(
    x=pd.Timestamp('2024-11-05').timestamp() * 1000,
    line_dash='dash',
    line_color='grey',
    annotation_text='Election Day',
)

fig_zoom.update_layout(
    title='Trump Win Probability — Election Week Detail',
    xaxis_title='Date',
    yaxis_title='Implied Probability',
    yaxis=dict(range=[0, 1], tickformat='.0%'),
    hovermode='x unified'
)

fig_zoom.show()

On the graph, it seems as if Polymarket jumped to 100% slightly after Kalshi, which gives the idea that profit could be made during this time. However, this is only due to the 12-hour granularity, as the next point on the graph is at November 6th, 12:00 PM. This shows that for events that resolve quickly like election night, 12-hour granularity might not be sufficient to measure convergence speed accurately.

We also see that on November 3rd, Kalshi dips to ~49% while Polymarket stayed at ~56%. This is a significant divergence that takes place before the election. We aim to analyze such divergences, and see if they are correlated to validator set concentration.

Use the Dune API to collect data of each validator stake for each date between Jan 1, 2024 and Feb 1, 2026. This data will be used to determine validator set concentration on a given day.

In [15]:
import time

API_KEY = os.getenv("DUNE_API_KEY")
QUERY_ID = 7613634 # Custom query made through Dune

headers = {"X-Dune-API-Key": API_KEY}

# execute the query
exec_resp = requests.post(
    f"https://api.dune.com/api/v1/query/{QUERY_ID}/execute",
    headers=headers
).json()

execution_id = exec_resp['execution_id']
print(f"execution_id: {execution_id}")

while True:
    status = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/status",
        headers=headers
    ).json()
    state = status['state']
    print(f"state: {state}")
    if state == 'QUERY_STATE_COMPLETED':
        break
    elif state == 'QUERY_STATE_FAILED':
        print("Query failed:", status)
        break
    time.sleep(5)

results = requests.get(
    f"https://api.dune.com/api/v1/execution/{execution_id}/results",
    headers=headers,
    params={"limit": 50000}
).json()

df = pd.DataFrame(results['result']['rows'])
print(df.shape)
print(df.head())

project_root = os.path.dirname(os.getcwd())
save_dir = os.path.join(project_root, 'data', 'raw', 'validators')

os.makedirs(save_dir, exist_ok=True)

df.to_csv(os.path.join(save_dir, 'polygon_validators.csv'), index=False)
print("Saved.")

execution_id: 01KT57YVAMB5W2DNHG6F7KNN4Z
state: QUERY_STATE_PENDING
state: QUERY_STATE_COMPLETED
(29912, 5)
         amount                         time                  validator  \
0  2.190883e+08  2026-02-28 00:00:00.000 UTC             Luganodes (18)   
1  4.650227e+07  2026-02-28 00:00:00.000 UTC  Staking4all - 0% fee (30)   
2  3.144817e+06  2026-02-28 00:00:00.000 UTC         AnkrValidator (31)   
3  3.710437e+07  2026-02-28 00:00:00.000 UTC             StakePool (32)   
4  3.157875e+06  2026-02-28 00:00:00.000 UTC             HashQuark (34)   

   validator_id        validator_name  
0            18             Luganodes  
1            30  Staking4all - 0% fee  
2            31         AnkrValidator  
3            32             StakePool  
4            34             HashQuark  
Saved.


Collect the price history for each event in events.csv using get_all_price_histories() from polymarket.py

In [19]:
from src.collect.polymarket import get_all_price_histories

results = get_all_price_histories()

trump_wins_2024: 614 records
harris_wins_2024: 613 records
fed_holds_march_2025: 182 records
fed_holds_september_2025: 266 records
fed_holds_december_2025: 262 records
bad_bunny_top_spotify_artist_2025: 239 records
govt_shutdown_by_jan_31_2026: 166 records
us_recession_in_2025: 716 records


Collect the price history for each event in events.csv using get_all_price_histories() from kalshi.py

In [20]:
from src.collect.kalshi import get_all_price_histories

results = get_all_price_histories()

trump_wins_2024: 109 records
harris_wins_2024: 109 records
fed_holds_march_2025: 49 records
fed_holds_september_2025: 50 records
fed_holds_december_2025: 73 records
bad_bunny_top_spotify_artist_2025: 212 records
govt_shutdown_by_jan_31_2026: 82 records
us_recession_in_2025: 513 records
